## Build a Risk Prediction Model for Carbon Emissions

### Load the Training Dataset into Unity Catalog Volume

In [ ]:
%sql

CREATE VOLUME IF NOT EXISTS carbonops.bronze.ml_raw_files;

In [ ]:
import requests

# Define the current catalog
catalog_name = "carbonops"

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/bronze/ml_raw_files"

# List of files to download
files = ["carbon_emissions_risk_dataset.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/Databricks-Job-Ready-Projects/refs/heads/main/Machine-Learning/Logistic-Regression-Model/Data/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Split the dataset into training and testing sets

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Load CarbonOps Risk Prediction Dataset
data = (
    spark.read.format("csv")
    .option("header", "true")
    .load("/Volumes/carbonops/bronze/ml_raw_files/carbonops_emissions_risk_classification.csv")
)

# Cast columns and exclude facility_id
data = data.select(
    col("year").cast("int"),
    col("month").cast("int"),
    col("electricity_kwh").cast("double"),
    col("diesel_litres").cast("double"),
    col("natural_gas_m3").cast("double"),
    col("lpg_kg").cast("double"),
    col("production_units").cast("double"),
    col("operating_hours").cast("double"),
    col("avg_temperature_c").cast("double"),
    col("current_emissions_tco2e").cast("double"),
    col("previous_month_emissions_tco2e").cast("double"),
    col("monthly_emissions_budget_tco2e").cast("double"),
    col("next_month_risk").cast("int")
).dropna()

# Split into Training and Testing Datasets (70/30)
splits = data.randomSplit([0.7, 0.3], seed=42)

train = splits[0]
test = splits[1]

print("Training Rows:", train.count(), "Testing Rows:", test.count())

### Define the Training Pipeline

In [ ]:
import mlflow
import mlflow.spark

from mlflow.models import infer_signature
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

def train_carbon_risk_model(training_data, test_data, maxIterations, regularization):

    with mlflow.start_run(run_name="CarbonOps_Logistic_Regression"):

        # Define numerical features
        numFeatures = [
            "year",
            "month",
            "electricity_kwh",
            "diesel_litres",
            "natural_gas_m3",
            "lpg_kg",
            "production_units",
            "operating_hours",
            "avg_temperature_c",
            "current_emissions_tco2e",
            "previous_month_emissions_tco2e",
            "monthly_emissions_budget_tco2e"
        ]

        # Assemble features
        numVector = VectorAssembler(
            inputCols=numFeatures,
            outputCol="numericFeatures"
        )

        # Scale features
        numScaler = MinMaxScaler(
            inputCol="numericFeatures",
            outputCol="normalizedFeatures"
        )

        # Logistic Regression
        mlAlgo = LogisticRegression(
            labelCol="next_month_risk",
            featuresCol="normalizedFeatures",
            maxIter=maxIterations,
            regParam=regularization
        )

        # Create ML Pipeline
        pipeline = Pipeline(stages=[
            numVector,
            numScaler,
            mlAlgo
        ])

        # Train the model
        print("Training CarbonOps Logistic Regression Model...")
        model = pipeline.fit(training_data)

        # Generate predictions
        predictions = model.transform(test_data)

        # Evaluate classification metrics
        evaluator = MulticlassClassificationEvaluator(
            labelCol="next_month_risk",
            predictionCol="prediction"
        )

        accuracy = evaluator.evaluate(
            predictions, {evaluator.metricName: "accuracy"}
        )

        precision = evaluator.evaluate(
            predictions, {evaluator.metricName: "weightedPrecision"}
        )

        recall = evaluator.evaluate(
            predictions, {evaluator.metricName: "weightedRecall"}
        )

        f1 = evaluator.evaluate(
            predictions, {evaluator.metricName: "f1"}
        )

        auc_evaluator = BinaryClassificationEvaluator(
            labelCol="next_month_risk",
            metricName="areaUnderROC"
        )

        auc = auc_evaluator.evaluate(predictions)

        # Print metrics
        print("Accuracy:", accuracy)
        print("Weighted Precision:", precision)
        print("Weighted Recall:", recall)
        print("F1 Score:", f1)
        print("AUC-ROC:", auc)

        # Log parameters
        mlflow.log_param("maxIter", maxIterations)
        mlflow.log_param("regParam", regularization)

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("weightedPrecision", precision)
        mlflow.log_metric("weightedRecall", recall)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("AUC_ROC", auc)

        # Create input example
        input_example = training_data.select(
            numFeatures
        ).limit(5).toPandas()

        # Infer model signature
        output_example = model.transform(
            spark.createDataFrame(input_example)
        ).select("prediction").toPandas()

        signature = infer_signature(
            input_example,
            output_example
        )

        # Log Spark ML model
        mlflow.spark.log_model(
            spark_model=model,
            artifact_path="model",
            signature=signature,
            input_example=input_example
        )

        print("CarbonOps Logistic Regression experiment completed!")

        return model

### Train the Logistic Regression Model

In [ ]:
trained_model = train_carbon_risk_model(
    training_data=train,
    test_data=test,
    maxIterations=100,
    regularization=0.01
)

### Test Model Prediction with Sample Data

In [ ]:
from pyspark.sql.functions import col

# Create sample data for risk prediction
newData = spark.createDataFrame([{
    "year": 2026,
    "month": 11,
    "electricity_kwh": 65000.0,
    "diesel_litres": 3200.0,
    "natural_gas_m3": 1500.0,
    "lpg_kg": 650.0,
    "production_units": 12000.0,
    "operating_hours": 550.0,
    "avg_temperature_c": 32.0,
    "current_emissions_tco2e": 210.0,
    "previous_month_emissions_tco2e": 195.0,
    "monthly_emissions_budget_tco2e": 200.0
}])

# Generate predictions using trained model
predictions = trained_model.transform(newData)

# Display predicted risk and probabilities
display(
    predictions.select(
        "current_emissions_tco2e",
        "monthly_emissions_budget_tco2e",
        col("prediction").alias("PredictedRisk"),
        "probability"
    )
)